## Imports

In [ ]:
import ...
import ...
import ...
import ...
import ...

## Data

In [ ]:
# TODO: load ECG5000 train and test arff files using scipy.io.arff.loadarff
# note: download from http://timeseriesclassification.com/aeon-toolkit/ECG5000.zip if needed
train_data, _ = ...
test_data, _ = ...

# TODO: convert structured array to numpy, separate features and label (last column)
train_np = ...
test_np = ...

In [ ]:
# TODO: combine train+test, features = all cols except last, labels = last col
X_all = ...
y_all = ...

# TODO: labels are 1-indexed, remap to 0-indexed
y_all = ...

assert X_all.shape[1] == 140
assert y_all.min() == 0

In [ ]:
# TODO: reshape X to (N, 140, 1) for LSTM input
X_all = ...

assert X_all.shape == (X_all.shape[0], 140, 1)

In [ ]:
# TODO: normalize per-feature (over N and T dims), split 80/20, to tensors, DataLoader batch=64
X_train, X_test, y_train, y_test = ...
train_loader = ...
test_loader = ...

batch_X, batch_y = next(iter(train_loader))
assert batch_X.shape == (64, 140, 1)
assert batch_X.dtype == torch.float32
assert batch_y.dtype == torch.long

## LSTM API

In [ ]:
# TODO: fill in all shapes BEFORE building any model
# nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
#
# input:  (B, 140, 1)
# output: (B, 140, ?)   <- what goes here?
# h_n:    (?, B, ?)     <- what goes here?
# c_n:    (?, B, ?)     <- what goes here?
#
# with bidirectional=True:
# output: (B, 140, ?)   <- what goes here?
# h_n:    (?, B, ?)     <- what goes here?

## Model

In [ ]:
# TODO: LSTM classifier — LSTM(1, 64, 2, batch_first=True) -> take last timestep -> Linear(64, 5)
class ECGClassifier(nn.Module):
    def __init__(self):
        ...
    def forward(self, x):
        # TODO: run lstm, take out[:, -1, :], pass through linear
        ...

model = ECGClassifier()
out = model(batch_X)
assert out.shape == (64, 5)

## Verify

In [ ]:
# TODO: prove out[:, -1, :] == h_n[-1] for unidirectional LSTM
lstm = nn.LSTM(1, 64, num_layers=2, batch_first=True)
x = batch_X
output, (h_n, c_n) = lstm(x)

last_output = output[:, -1, :]
last_hidden = h_n[-1]

# they match because h_n[-1] is the hidden state of the last layer at the last timestep
assert torch.allclose(last_output, last_hidden, atol=1e-5)

## Bidirectional

In [ ]:
# TODO: bidirectional LSTM classifier — fix Linear input size
class ECGBiLSTM(nn.Module):
    def __init__(self):
        # TODO: LSTM(1, 64, 2, batch_first=True, bidirectional=True)
        # TODO: Linear(?, 5) — what should ? be?
        ...
    def forward(self, x):
        # TODO: concat forward last timestep and backward first timestep from h_n
        ...

model_bi = ECGBiLSTM()
out_bi = model_bi(batch_X)
assert out_bi.shape == (64, 5)

In [ ]:
# TODO: verify h_n shape for bidirectional
lstm_bi = nn.LSTM(1, 64, num_layers=2, batch_first=True, bidirectional=True)
output_bi, (h_n_bi, c_n_bi) = lstm_bi(batch_X)

# note: h_n shape is (num_layers * num_directions, B, hidden_size)
assert h_n_bi.shape == (4, 64, 64)
assert output_bi.shape == (64, 140, 128)

## Training

In [ ]:
model = ECGClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 30
max_grad_norm = 1.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        # TODO: forward, loss, backward, clip_grad_norm_, step, zero_grad
        ...

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # TODO: accumulate correct
            ...

    if epoch % 5 == 0:
        print(f"epoch {epoch}: loss={total_loss/len(train_loader):.4f} acc={correct/total:.4f}")

assert correct / total > 0.85, f"accuracy too low: {correct/total}"

## Understanding

In [ ]:
# Q1: why does h_n have shape (num_layers * num_directions, B, H) instead of (num_layers, num_directions, B, H)?
# A1: ...

# Q2: what happens if you feed a sequence shorter than 140 steps to this trained model?
# A2: ...

# Q3: why do we clip gradients for RNNs but usually not for MLPs?
# A3: ...